<a href="https://colab.research.google.com/github/jdklii22/stock/blob/main/pivot_cli.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📈 Stock Pivot Points Calculator (Google Colab Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jdklii22/pivot/blob/main/pivot_cli.ipynb)

This notebook calculates **Standard Floor Pivot Points** across **Monthly, Quarterly, and Annual** completed periods, displays whole-number percentage distances from the current live price, and automatically exports formatted Markdown summaries.

In [12]:
#@title 📦 Step 1: Install & Import Dependencies
# Run this cell first to set up yfinance, pandas, plotly, and kaleido for static image export
!pip install -q yfinance pandas plotly kaleido

import os
import sys
import warnings
from datetime import datetime

# Suppress internal warnings
os.environ['PYTHONWARNINGS'] = 'ignore'
warnings.simplefilter('ignore')

import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from IPython.display import display, HTML, Markdown

print('✅ Dependencies loaded successfully!')

✅ Dependencies loaded successfully!


In [13]:
# Step 2: Define Core Calculation & Formatting Engine

LEVEL_NAMES = [
    ("Resistance 3 (R3)", "R3"),
    ("Resistance 2 (R2)", "R2"),
    ("Resistance 1 (R1)", "R1"),
    ("PIVOT POINT (P)",   "P"),
    ("Support 1 (S1)",    "S1"),
    ("Support 2 (S2)",    "S2"),
    ("Support 3 (S3)",    "S3"),
]

def calculate_timeframe_pivots(df, resample_code):
    """
    Resample daily historical dataframe to the given frequency code,
    extract prior completed period (iloc[-2]), and calculate standard floor pivots.
    """
    resampled = df.resample(resample_code).agg(
        {"High": "max", "Low": "min", "Close": "last"}
    ).dropna()

    if len(resampled) < 2:
        return None, "Insufficient completed historical periods."

    prev_period = resampled.iloc[-2]
    H = float(prev_period["High"])
    L = float(prev_period["Low"])
    C = float(prev_period["Close"])
    period_label = str(prev_period.name.strftime('%Y-%m-%d') if hasattr(prev_period.name, 'strftime') else prev_period.name)

    P = (H + L + C) / 3.0
    R1 = (P * 2.0) - L
    S1 = (P * 2.0) - H
    R2 = P + (R1 - S1)
    S2 = P - (R1 - S1)
    R3 = H + 2.0 * (P - L)
    S3 = L - 2.0 * (H - P)

    return {
        "H": H, "L": L, "C": C, "period_label": period_label,
        "pivots": {
            "R3": R3, "R2": R2, "R1": R1, "P": P,
            "S1": S1, "S2": S2, "S3": S3
        }
    }, None

def create_candlestick_chart(symbol, df, current_price, pivot_data, timeframe_label, lookback_days=250):
    """Render an interactive Plotly candlestick chart with specific pivot levels."""
    sub_df = df.tail(lookback_days)
    fig = go.Figure(data=[
        go.Candlestick(
            x=sub_df.index,
            open=sub_df['Open'],
            high=sub_df['High'],
            low=sub_df['Low'],
            close=sub_df['Close'],
            name=symbol,
            increasing_line_color='#22c55e',
            decreasing_line_color='#ef4444'
        )
    ])

    # Add Pivot overlay for specified timeframe
    if pivot_data:
        pivots = pivot_data['pivots']
        color_map = {
            'Monthly': {'P': '#38bdf8', 'R1': '#f87171', 'S1': '#86efac'},
            'Quarterly': {'P': '#c084fc', 'R1': '#f472b6', 'S1': '#fbbf24'},
            'Annual': {'P': '#2dd4bf', 'R1': '#fb7185', 'S1': '#a7f3d0'}
        }
        colors = color_map.get(timeframe_label, {'P': '#38bdf8', 'R1': '#f87171', 'S1': '#86efac'})

        fig.add_hline(y=pivots['P'], line_dash='dashdot', line_color=colors['P'], line_width=2,
                      annotation_text=f"{timeframe_label} P: ${pivots['P']:.2f}", annotation_position="top right")
        fig.add_hline(y=pivots['R1'], line_dash='dash', line_color=colors['R1'], line_width=1,
                      annotation_text=f"{timeframe_label} R1: ${pivots['R1']:.2f}", annotation_position="top right")
        fig.add_hline(y=pivots['S1'], line_dash='dash', line_color=colors['S1'], line_width=1,
                      annotation_text=f"{timeframe_label} S1: ${pivots['S1']:.2f}", annotation_position="bottom right")

    # Current Price Line
    fig.add_hline(y=current_price, line_dash='dot', line_color='#eab308', line_width=1.5,
                  annotation_text=f"Current: ${current_price:.2f}", annotation_position="bottom left")

    fig.update_layout(
        title=f"<b>{symbol}</b> — Recent Price Action & {timeframe_label} Pivot Levels",
        xaxis_rangeslider_visible=False,
        template='plotly_dark',
        height=500,
        margin=dict(l=40, r=90, t=50, b=40),
        hovermode='x unified'
    )
    return fig

def export_markdown(path, symbol, current_price, latest_date, rows, m_data, q_data, a_data):
    """Generate a clean, professional Markdown report."""
    now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    md = [
        f"# {symbol} — Stock Pivot Points Analysis", "",
        f"> **Generated:** {now_str}  ",
        f"> **Current Market Price:** `${current_price:.2f}`  ",
        f"> **Market Data As Of:** `{latest_date}`  ", "",
        "## Pivot Points Summary (Floor Model)", "",
        "| Level | Monthly Price (% Dist) | Quarterly Price (% Dist) | Annual Price (% Dist) |",
        "| :--- | :--- | :--- | :--- |"
    ]
    for r in rows:
        name, m_txt, q_txt, a_txt = r["Level"], r["Monthly"]["text"], r["Quarterly"]["text"], r["Annual"]["text"]
        if r["Code"] == "P":
            md.append(f"| **{name}** | **{m_txt}** | **{q_txt}** | **{a_txt}** |")
        else:
            md.append(f"| {name} | {m_txt} | {q_txt} | {a_txt} |")
    md.extend([
        "", "---", "",
        "## Underlying Calculation Data (Previous Completed Periods)", "",
        "| Period | Completed Period End | High | Low | Close |",
        "| :--- | :--- | :--- | :--- | :--- |"
    ])
    for label, d in [("Monthly", m_data), ("Quarterly", q_data), ("Annual", a_data)]:
        if d:
            md.append(f"| **{label}** | {d['period_label']} | `${d['H']:.2f}` | `${d['L']:.2f}` | `${d['C']:.2f}` |")
        else:
            md.append(f"| **{label}** | *N/A* | *N/A* | *N/A* | *N/A* |")
    md.append("")
    content = "\n".join(md)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    return content

def process_ticker(symbol, download_md=False, show_chart=True):
    """Process ticker, compute pivots, render HTML table, charts, save static images, and export Markdown."""
    symbol = symbol.strip().upper()
    print(f"⏳ Fetching 5-year market data for {symbol}...")
    ticker = yf.Ticker(symbol)
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            df = ticker.history(period="5y")
    except Exception as e:
        print(f"❌ Error fetching data: {e}")
        return

    if df is None or df.empty:
        print(f"❌ No price data found for ticker '{symbol}'. Please check symbol.")
        return

    current_price = float(df["Close"].iloc[-1])
    latest_date = df.index[-1].strftime("%Y-%m-%d")

    monthly_data, m_err = calculate_timeframe_pivots(df, "ME")
    quarterly_data, q_err = calculate_timeframe_pivots(df, "QE")
    annual_data, a_err = calculate_timeframe_pivots(df, "YE")

    rows = []
    for full_name, code in LEVEL_NAMES:
        row = {"Level": full_name, "Code": code}
        for tf_name, tf_data in [("Monthly", monthly_data), ("Quarterly", quarterly_data), ("Annual", annual_data)]:
            if tf_data and code in tf_data["pivots"]:
                price = tf_data["pivots"].get(code)
                dist_pct = round(((price / current_price) - 1.0) * 100.0)
                row[tf_name] = {"price": price, "dist_pct": dist_pct, "text": f"${price:.2f} ({dist_pct:+d}%)"}
            else:
                row[tf_name] = {"price": 0.0, "dist_pct": 0, "text": "N/A"}
        rows.append(row)

    # Render Rich HTML Table
    table_html = f"""
    <div style='background: #1e293b; padding: 1.25rem; border-radius: 8px; font-family: sans-serif; color: #f1f5f9; margin-bottom: 1rem;'>
      <h3 style='margin-top: 0; color: #38bdf8;'>📈 {symbol} Pivot Points</h3>
      <p style='color: #94a3b8; font-size: 0.95rem;'>Current Price: <b style='color: #34d399;'>${current_price:.2f}</b> &nbsp;|&nbsp; Market Date: <code>{latest_date}</code></p>
      <table style='width: 100%; border-collapse: collapse; text-align: left; background: #0f172a; border-radius: 6px; overflow: hidden;'>
        <thead>
          <tr style='background: #334155; color: #38bdf8;'>
            <th style='padding: 8px 12px;'>Level</th>
            <th style='padding: 8px 12px;'>Monthly Price (% Dist)</th>
            <th style='padding: 8px 12px;'>Quarterly Price (% Dist)</th>
            <th style='padding: 8px 12px;'>Annual Price (% Dist)</th>
          </tr>
        </thead>
        <tbody>
    """
    for r in rows:
        code = r['Code']
        bg = 'rgba(239, 68, 68, 0.15)' if code.startswith('R') else ('rgba(56, 189, 248, 0.2)' if code == 'P' else 'rgba(34, 197, 94, 0.15)')
        color = '#f87171' if code.startswith('R') else ('#38bdf8' if code == 'P' else '#4ade80')
        is_bold = 'font-weight: bold;' if code == 'P' else ''
        table_html += f"<tr style='border-bottom: 1px solid #1e293b; background: {bg}; {is_bold}'>"
        table_html += f"<td style='padding: 8px 12px; color: {color};'>{r['Level']}</td>"
        table_html += f"<td style='padding: 8px 12px;'>{r['Monthly']['text']}</td>"
        table_html += f"<td style='padding: 8px 12px;'>{r['Quarterly']['text']}</td>"
        table_html += f"<td style='padding: 8px 12px;'>{r['Annual']['text']}</td></tr>"
    table_html += "</tbody></table></div>"
    display(HTML(table_html))

    # Export Markdown file
    md_path = f"{symbol}_pivots.md"
    export_markdown(md_path, symbol, current_price, latest_date, rows, monthly_data, quarterly_data, annual_data)
    print(f"💾 Saved report to '{md_path}'")

    # Optional Colab file download
    if download_md:
        try:
            from google.colab import files
            files.download(md_path)
            print(f"📥 Initiated browser download for {md_path}")
        except Exception as e:
            print(f"(Colab file download available when running in Google Colab environment)")

    # Generate and Optional display/save Candlestick Charts
    print("📊 Generating charts...")
    fig_monthly = create_candlestick_chart(symbol, df, current_price, monthly_data, 'Monthly')
    fig_quarterly = create_candlestick_chart(symbol, df, current_price, quarterly_data, 'Quarterly')
    fig_annual = create_candlestick_chart(symbol, df, current_price, annual_data, 'Annual')

    try:
        # Save static image exports
        fig_monthly.write_image(f"{symbol}_monthly.png")
        fig_quarterly.write_image(f"{symbol}_quarterly.png")
        fig_annual.write_image(f"{symbol}_annual.png")
        print(f"💾 Static chart images saved to disk as: '{symbol}_monthly.png', '{symbol}_quarterly.png', and '{symbol}_annual.png'")
    except Exception as image_err:
        print(f"⚠️ Could not save static images. Ensure 'kaleido' dependency is fully loaded. Error: {image_err}")

    if show_chart:
        print("📊 Plotting Monthly Pivot Levels...")
        fig_monthly.show()

        print("📊 Plotting Quarterly Pivot Levels...")
        fig_quarterly.show()

        print("📊 Plotting Annual Pivot Levels...")
        fig_annual.show()

## 🎯 Method 1: Google Colab Interactive Form (Recommended)
Adjust the fields below and run the cell to calculate pivot points instantly.

In [17]:
#@title 🔍 Run Pivot Calculator { run: "auto" }
ticker_symbol = "spy" #@param {type:"string"}
show_candlestick_chart = True #@param {type:"boolean"}
download_markdown_file = False #@param {type:"boolean"}

if ticker_symbol.strip():
    process_ticker(ticker_symbol.strip(), download_md=download_markdown_file, show_chart=show_candlestick_chart)
else:
    print("Please enter a valid ticker symbol.")

⏳ Fetching 5-year market data for SPY...


Level,Monthly Price (% Dist),Quarterly Price (% Dist),Annual Price (% Dist)
Resistance 3 (R3),$806.03 (+10%),$834.43 (+14%),$963.00 (+31%)
Resistance 2 (R2),$793.83 (+8%),$805.93 (+10%),$824.65 (+13%)
Resistance 1 (R1),$774.77 (+6%),$784.28 (+7%),$750.64 (+2%)
PIVOT POINT (P),$762.57 (+4%),$755.79 (+3%),$612.29 (-16%)
Support 1 (S1),$743.51 (+1%),$734.14 (+0%),$538.28 (-27%)
Support 2 (S2),$731.31 (+0%),$705.64 (-4%),$399.93 (-45%)
Support 3 (S3),$712.25 (-3%),$683.99 (-7%),$325.92 (-56%)


💾 Saved report to 'SPY_pivots.md'
📊 Generating charts...
⚠️ Could not save static images. Ensure 'kaleido' dependency is fully loaded. Error: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido

📊 Plotting Monthly Pivot Levels...


📊 Plotting Quarterly Pivot Levels...


📊 Plotting Annual Pivot Levels...


## 💻 Method 2: Interactive Prompt Loop (CLI Style)
Run this cell to use the terminal-style continuous prompt. Type `q` or `quit` to exit.

In [ ]:
# Interactive CLI Loop
while True:
    try:
        user_input = input("\nEnter ticker symbol ('q' or 'quit' to exit): ").strip()
        if not user_input:
            continue
        if user_input.lower() in ('q', 'quit', 'exit'):
            print("Exiting. Have a profitable trading day! 👋")
            break
        process_ticker(user_input, download_md=False, show_chart=False)
    except (KeyboardInterrupt, EOFError):
        print("\nExiting.")
        break

## 📊 Method 3: Watchlist Batch Analysis
Compare multiple stocks or ETFs across your watchlist in a single run.

In [ ]:
# Batch Watchlist
watchlist = ["QQQ", "SPY", "AAPL", "NVDA", "TSLA"]

for sym in watchlist:
    process_ticker(sym, download_md=False, show_chart=False)